# Tarea grupal: "EnviRápido", centro de distribución de paquetes


**Grupos de 5 · Lenguaje: Python

Este cuaderno contiene el **enunciado**, los **mínimos necesarios**, los **puntos de libertad**, los **entregables** y la **rúbrica**. Al final hay un espacio de trabajo con una celda de código por cada mínimo.



---
# 1. El problema
Una empresa de mensajería, **EnviRápido**, tiene un centro de distribución que hoy funciona con cuadernos y hojas de papel. Su dueño sufre estos problemas:

- Cada día llegan cientos de paquetes y se atienden **en el orden en que llegaron**, pero a veces los empleados se saltan paquetes sin darse cuenta.
- Los clientes llaman preguntando *"¿dónde está mi paquete?"* y el empleado tarda minutos en encontrarlo revisando hoja por hoja.
- A veces un empleado marca un paquete como "entregado" por error y no hay forma de corregirlo ordenadamente.
- La empresa reparte a **zonas/sucursales** (cientos de ellas, con un código numérico) y necesita saber rápidamente a cuál corresponde cada código.
- Hay paquetes que llegan **sin código de seguimiento**, solo con el nombre del cliente, y también hay que poder ubicarlos.

**Tu grupo es una empresa de software** que debe ofrecerle al dueño un prototipo que resuelva estos problemas. Ustedes deciden qué datos tiene un paquete, cómo se organiza el sistema y qué estructuras usar. El dueño no sabe de programación: tendrán que **convencerlo** de que su solución es buena y de por qué.

---
# 2. Mínimos necesarios (para aprobar)

1. **Modelado propio:** definir qué es un paquete, un cliente y una zona (clases o estructuras), y explicar por escrito qué decidieron y por qué.
2. **Recepción y atención en orden de llegada**, con posibilidad de ver quién sigue.
3. **Consulta rápida por código de seguimiento**, usando **una tabla hash implementada por ustedes** (sin usar `dict` ni `set` de Python para la estructura principal), con **manejo de colisiones**.
4. **Corrección de errores** (deshacer acciones, al menos las últimas) usando una estructura adecuada.
5. **Búsqueda de zona por código** sobre una lista ordenada, usando búsqueda binaria.
6. **Búsqueda de paquetes por nombre de cliente** (sin código), usando búsqueda secuencial.
7. **Pruebas:** un programa de ejemplo con al menos 20 paquetes que demuestre cada funcionalidad, incluyendo casos límite (cola vacía, código inexistente, colisión, deshacer sin historial).
8. **Medición:** comparar cuántas comparaciones hace cada búsqueda (secuencial vs. binaria vs. hash) con 100, 1000 y 10000 datos, y mostrar los resultados.

---
# 3. Puntos de libertad (donde tienen que pensar)
Estas decisiones **no las da el enunciado**; deben tomarlas y justificarlas:

- ¿Qué pasa con los paquetes urgentes o con prioridad? ¿Rompen la regla del orden de llegada?
- ¿Qué se hace si dos paquetes tienen el mismo nombre de cliente?
- ¿Cuál es el tamaño de la tabla hash y qué función hash usan? ¿Qué hacen cuando se llena?
- ¿Qué acciones se pueden deshacer y cuáles no?
- ¿La lista de zonas puede cambiar? Si cambia, ¿cómo se mantiene ordenada?
- ¿Cómo reportan al usuario que algo no se encontró?

**Extras opcionales (para nota adicional):** reasignar paquetes entre zonas, estadísticas del día, interfaz de menú por consola, carga de datos desde archivo.

---
# 4. Entregables y "venta"

| Entregable | Detalle |
|---|---|
| Código | Este cuaderno `.ipynb` (o archivos `.py`), comentado y ordenado |
| **Presentación de venta (10 min )** | Demostración en vivo + argumento de por qué su solución es la mejor |

**Reglas de la "venta":** los 5 integrantes deben hablar. El público (el resto de la clase y el docente) actúa como el dueño de la empresa y puede hacer preguntas como *"¿qué pasa si hay 1 millón de paquetes?"* o *"¿por qué no usaron una lista para esto?"*.



---
# ESPACIO DE TRABAJO DEL GRUPO

### Equipo y roles
Completen esta tabla (haciendo doble clic en la celda). Los roles deben **rotar** al preparar la defensa.

| Integrante | Rol principal | Rol de la defensa |
|---|---|---|
|José Checa|Programador 1|Expocisión técnica|
|Symond Salazar|Programador 2|Responder preguntas 1|
|Erick Obando|Encargado de presentación 1|Exponer ventajas|
|David Carrero|Encargado del guión|Exponer la idea inicial|
|María Ramírez|Encargada de presentación 2|Expone los problemas del sistema anterior|

### Registro de decisiones de diseño
Escriban aquí, en pocas líneas, qué decidieron para cada punto de libertad y **por qué** (esto alimenta el informe).

- **Paquetes urgentes:** Se manejarán con dos colas separadas (una de prioridad "urgente" y otra "normal"). Siempre se vacía primero la cola urgente, respetando su orden de llegada
- **Nombres de cliente repetidos:** La función de búsqueda secuencial no se detendrá en la primera coincidencia; recorrerá toda la estructura y devolverá una lista con todos los paquetes asociados a ese nombre
- **Tamaño y función hash / tabla llena:** Se usará un tamaño inicial basado en un número primo para reducir coincidencias. La función hash sumará los valores de los caracteres del código de seguimiento y aplicará valor absoluto sobre el tamaño. Como se usan listas dentro de listas, se debe aplicar rehashing para evitar que se vuelva lenta.
- **Qué se puede deshacer y qué no:** Solo se podrán deshacer los cambios de estado. No se deshará la recepción de un paquete ya que físicamente el paquete ya ingresó al almacén
- **Zonas (¿cambian? ¿cómo se mantienen ordenadas?):** Las zonas se cargarán en una lista que se ordenará una sola vez al inicio del programa, optimizando la búsqueda binaria
- **Cómo se informa que algo no se encontró:** Las funciones retornarán None o una lista vacía []. Se interpretarán como "Paquete no encontrado"

### Preparación: módulos
Importen aquí lo que necesiten (por ejemplo `randint` para generar datos de prueba). **Recuerden:** no usar `dict` ni `set` para la tabla hash principal.

In [1]:
from random import randint,choice  # agreguen aquí los módulos que necesiten
import time
import string

### Mínimo 1 · Modelado: paquete, cliente y zona
Definan las clases o estructuras. Antes de programar, escriban en papel los **sustantivos** y los **verbos** del problema.

**Decisión de modelado**: Se definieron 3 clases independientes (Cliente, Paquete y Zona). Paquete contiene una instancia de Cliente para acceder directo a sus datos, un bool urgente para la prioridad en las colas, y el atributo estado que inicia 'En centro' para rastrear su ciclo de vida

In [2]:
# Tu código aquí: clases Paquete, Cliente, Zona
class Cliente:
  def __init__(self,nombre,telefono=""):
    self.nombre=nombre
    self.telefono=telefono

class Paquete:
  def __init__(self,codigo,cliente,zona,urgente=False):
    self.codigo=codigo
    self.cliente=cliente
    self.zona=zona
    self.urgente=urgente
    # Todo paquete inicia en la bodega central al ser registrado
    self.estado="En centro"

  def __repr__(self):
    # Define cómo se muestra el paquete en pantalla al hacdr print()
    return f"[{self.codigo}] {self.cliente.nombre} - Zona: {self.zona} ({self.estado})"

class Zona:
  def __init__(self,codigo,sucursal):
    self.codigo = codigo
    self.sucursal = sucursal

### Mínimo 2 · Recepción y atención en orden de llegada
Necesitan: recibir un paquete, atender el siguiente y **ver quién sigue** sin sacarlo.

In [3]:
# Tu código aquí: estructura para el orden de llegada
class CentroDeDistribucion:
  def __init__(self):
    # Dos colas FIFO separadas para priorizar urgentes sin
    #perder el orden de llegada
    self.cola_normal=[]
    self.cola_urg=[]

  def recibir_paquete(self,paquete):
    if paquete.urgente:
      self.cola_urg.append(paquete)
    else:
      self.cola_normal.append(paquete)

  def ver_siguiente(self):
    # Retorna el elemento en la posicion [0] (el más antiguo)
    # sin eliminarlo de la lista
    if self.cola_urg:
      return self.cola_urg[0]
    elif self.cola_normal:
      return self.cola_normal[0]
    return None

  def atender(self):
    # Extrae y elimina con pop(0) el primer paquete urgente;
    # SOlo si está vacía pasa a la normal
    if self.cola_urg:
      return self.cola_urg.pop(0)
    elif self.cola_normal:
      return self.cola_normal.pop(0)
    return None

### Mínimo 3 · Consulta por código de seguimiento (tabla hash propia)
Implementen la tabla hash completa: función hash, insertar, buscar y manejo de colisiones. **Sin `dict` ni `set`.**

In [4]:
class TablaHash:
    def __init__(self, capacidad=23):
        self.capacidad = capacidad
        self.tabla = [[] for _ in range(capacidad)]
        self.comparaciones = 0

    def _hash(self, codigo):
        return abs(
            sum(ord(caracter) for caracter in str(codigo))
        ) % self.capacidad

    def insertar(self, codigo, paquete):
        cubeta = self.tabla[self._hash(codigo)]

        for posicion, (clave, valor) in enumerate(cubeta):
            if clave == codigo:
                cubeta[posicion] = (codigo, paquete)
                return

        cubeta.append((codigo, paquete))

    def buscar(self, codigo):
        self.comparaciones = 0

        for clave, paquete in self.tabla[self._hash(codigo)]:
            self.comparaciones += 1

            if clave == codigo:
                return paquete

        return None

    def generar_codigo_automatico(self, prefijo="AUTO"):
        contador = 1

        while True:
            codigo = f"{prefijo}-{contador:05d}"

            if self.buscar(codigo) is None:
                return codigo

            contador += 1

### Mínimo 4 · Deshacer acciones
Decidan qué acciones se registran y **qué información hay que guardar** para poder revertirlas.

In [5]:
# Tu código aquí: historial y deshacer
class Historial:
  def __init__(self):
    # Estructura LIFO: el último cambio guardado será el primero en deshacerse
    self.pila = []

  def cambiar_estado_paquete(self,paquete,nuevo_estado):
    estado_anterior = paquete.estado
    paquete.estado = nuevo_estado

    accion = (paquete,estado_anterior,nuevo_estado)

    self.pila.append(accion)
    print(f"Paquete {paquete.codigo} marcado como '{nuevo_estado}'")

  def deshacer_ultima_accion(self):
    # Caso límite: Evita un error si intentan deshacer cuando no hay historial
    if len(self.pila)==0:
      print("No hay acciones para deshacer")
      return None
    # pop() sin argumentos extrae el último elemento agregado al final de la lista
    paquete,estado_anterior,_=self.pila.pop()
    paquete.estado = estado_anterior

    print(f"Acción deshecha: El paquete {paquete.codigo} volvio al estado '{paquete.estado}'")

### Mínimo 5 · Búsqueda de zona por código (búsqueda binaria)
La lista de zonas debe estar **ordenada** antes de buscar.

In [6]:
def ordenar_zonas(zonas):
    # Requisito obligatorio para la búsqueda binaria: ordenar previamente por
    # código numérico
    return sorted(zonas, key=lambda zona: zona.codigo)


def buscar_zona_binaria(zonas_ordenadas, codigo):
    izquierda = 0
    derecha = len(zonas_ordenadas) - 1
    comparaciones = 0

    while izquierda <= derecha:
        # Calcula el punto central y descarta la mitad donde seguro no está
        medio = (izquierda + derecha) // 2
        comparaciones += 1

        if zonas_ordenadas[medio].codigo == codigo:
            return zonas_ordenadas[medio], comparaciones

        if zonas_ordenadas[medio].codigo < codigo:
            izquierda = medio + 1
        else:
            derecha = medio - 1

    return None, comparaciones

### Mínimo 6 · Búsqueda de paquetes por nombre de cliente (búsqueda secuencial)
Piensen qué devuelve su búsqueda si hay varios paquetes con el mismo nombre.

In [7]:
def buscar_paquetes_por_nombre(paquetes, nombre):
    # Recorre toda la lista sin detenerse en el primero, por si el cliente tiene
    # varios paquetes, usa .lower() en ambos lados para ignorar mayusculas y minusculas
    encontrados = []
    comparaciones = 0

    for paquete in paquetes:
        comparaciones += 1

        if paquete.cliente.nombre.lower() == nombre.lower():
            encontrados.append(paquete)

    return encontrados, comparaciones

### Mínimo 7 · Pruebas con al menos 20 paquetes
Demuestren **cada** funcionalidad. Incluyan casos límite: cola vacía, código inexistente, colisión y deshacer sin historial.

In [8]:
centro = CentroDeDistribucion()
historial = Historial()
tabla = TablaHash(23)
paquetes = []

# Generación de 20 paquetes: índices de 3 a 13 y repiten el nombre "Ana"
for i in range(20):
    nombre = "Ana" if i in (3, 13) else f"Cliente {i}"
    cliente = Cliente(nombre)

    paquete = Paquete(
        codigo=f"PKT{i:02d}",
        cliente=cliente,
        zona=(i % 5) + 1,
        urgente=(i in (5, 15))
    )

    paquetes.append(paquete)
    centro.recibir_paquete(paquete)
    tabla.insertar(paquete.codigo, paquete)


print("Siguiente paquete:", centro.ver_siguiente())
print("Paquete atendido:", centro.atender())

print("Búsqueda existente:", tabla.buscar("PKT03"))
print("Código inexistente:", tabla.buscar("NO-EXISTE"))

# Colisión intencional
# AB y BA producen la misma suma de caracteres
colision = TablaHash(1)
colision.insertar("AB", paquetes[0])
colision.insertar("BA", paquetes[1])

print("Colisión AB:", colision.buscar("AB"))
print("Colisión BA:", colision.buscar("BA"))


zonas = ordenar_zonas([
    Zona(3, "Sucursal Sur"),
    Zona(1, "Sucursal Centro"),
    Zona(2, "Sucursal Norte")
])

zona, comparaciones = buscar_zona_binaria(zonas, 2)
print("Zona encontrada:", zona.sucursal)
print("Comparaciones binarias:", comparaciones)

zona_inexistente, _ = buscar_zona_binaria(zonas, 99)
print("Zona inexistente:", zona_inexistente)

# Búsqueda secuencial por nombre
paquetes_ana, comparaciones = buscar_paquetes_por_nombre(
    paquetes,
    "Ana"
)

print("Paquetes de Ana:", paquetes_ana)
print("Comparaciones secuenciales:", comparaciones)

paquetes_nadie, _ = buscar_paquetes_por_nombre(
    paquetes,
    "Nadie"
)

print("Cliente inexistente:", paquetes_nadie)

# Deshacer cambio de estado
historial.cambiar_estado_paquete(
    paquetes[0],
    "En ruta"
)

historial.deshacer_ultima_accion()

# Segunda llamada consecutiva para demostrar el caso límite de pila vacía
historial.deshacer_ultima_accion()

# Vacíamos todas las colas para probar el caso límite de atender con el
# centro vacío
while centro.atender() is not None:
    pass

print("Cola vacía:", centro.atender())

cliente = Cliente("Carlos")

codigo = tabla.generar_codigo_automatico()

paquete = Paquete(
    codigo,
    cliente,
    zona=2
)

paquetes.append(paquete)
tabla.insertar(codigo, paquete)

print("Código generado:", codigo)
print("Paquete encontrado:", tabla.buscar(codigo))

Siguiente paquete: [PKT05] Cliente 5 - Zona: 1 (En centro)
Paquete atendido: [PKT05] Cliente 5 - Zona: 1 (En centro)
Búsqueda existente: [PKT03] Ana - Zona: 4 (En centro)
Código inexistente: None
Colisión AB: [PKT00] Cliente 0 - Zona: 1 (En centro)
Colisión BA: [PKT01] Cliente 1 - Zona: 2 (En centro)
Zona encontrada: Sucursal Norte
Comparaciones binarias: 1
Zona inexistente: None
Paquetes de Ana: [[PKT03] Ana - Zona: 4 (En centro), [PKT13] Ana - Zona: 4 (En centro)]
Comparaciones secuenciales: 20
Cliente inexistente: []
Paquete PKT00 marcado como 'En ruta'
Acción deshecha: El paquete PKT00 volvio al estado 'En centro'
No hay acciones para deshacer
Cola vacía: None
Código generado: AUTO-00001
Paquete encontrado: [AUTO-00001] Carlos - Zona: 2 (En centro)


### Mínimo 8 · Medición de comparaciones
Midan cuántas comparaciones hace cada búsqueda (secuencial, binaria y hash) con **100, 1000 y 10000** datos y muestren una tabla con los resultados. Después escriban en la celda siguiente qué observan.

In [9]:
def es_primo(numero):
    if numero < 2:
        return False

    divisor = 2

    while divisor * divisor <= numero:
        if numero % divisor == 0:
            return False

        divisor += 1

    return True


def siguiente_primo(numero):
    while not es_primo(numero):
        numero += 1

    return numero


resultados = []

for cantidad in (100, 1000, 10000):
    nombres = [
        f"Cliente {i}"
        for i in range(cantidad)
    ]

    paquetes_medicion = [
        Paquete(
            f"M{i}",
            Cliente(nombres[i]),
            1
        )
        for i in range(cantidad)
    ]

    # Búsqueda secuencial
    # Peor caso secuencial y binario: buscamos el último elemento para medir
    # el esfuerzo máximo
    _, comparaciones_secuenciales = buscar_paquetes_por_nombre(
        paquetes_medicion,
        nombres[-1]
    )

    # Búsqueda binaria
    zonas_medicion = ordenar_zonas([
        Zona(i, f"Sucursal {i}")
        for i in range(cantidad)
    ])

    _, comparaciones_binarias = buscar_zona_binaria(
        zonas_medicion,
        cantidad - 1
    )

    # Búsqueda hash
    # Dimensionamos la tabla hash al doble de datos (número primo)
    # para mantener bajas las colisiones
    capacidad = siguiente_primo(cantidad * 2 + 1)
    hash_medicion = TablaHash(capacidad)

    for paquete in paquetes_medicion:
        hash_medicion.insertar(
            paquete.codigo,
            paquete
        )

    hash_medicion.buscar(
        paquetes_medicion[-1].codigo
    )

    resultados.append([
        cantidad,
        comparaciones_secuenciales,
        comparaciones_binarias,
        hash_medicion.comparaciones
    ])


print("Datos | Secuencial | Binaria | Hash")

for fila in resultados:
    print(
        f"{fila[0]:5} | "
        f"{fila[1]:10} | "
        f"{fila[2]:7} | "
        f"{fila[3]:4}"
    )

Datos | Secuencial | Binaria | Hash
  100 |        100 |       7 |    1
 1000 |       1000 |      10 |    1
10000 |      10000 |      14 |    1


### Análisis de las mediciones
¿Qué pasa con las comparaciones al multiplicar los datos por 10? ¿Cuál búsqueda crece más rápido? ¿Su tabla hash se comportó como esperaban?

*Al multiplicar los datos por 10, la búsqueda secuencial también aumenta aproximadamente 10 veces, porque revisa los elementos uno por uno. La búsqueda binaria crece mucho más lentamente, ya que divide la lista ordenada a la mitad en cada comparación. La tabla hash fue la búsqueda más rápida y se comportó como esperábamos, realizando normalmente una comparación. Si ocurre una colisión, debe revisar los elementos de la misma posición o cubeta.*

### Lista de verificación antes de entregar
- [ ] Los 8 mínimos funcionan y están demostrados
- [ ] La tabla hash no usa `dict` ni `set` y maneja colisiones
- [ ] El código está comentado y ordenado
- [ ] Los 5 integrantes pueden explicar **cualquier** parte, no solo la suya
